In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn
from warnings import filterwarnings
import re

filterwarnings("ignore")
df = pd.read_csv("../data/interim/data.csv")

# Preprocessing

## Get Location info from Text

In [2]:
df["Text Text"].str.extract(r'Location:\s*([^,]+),\s*([^,]+),\s*([^,]+)')
df[["loc_1","loc_2","loc_3"]] = df["Text Text"].str.extract(r'Location:\s*([^,]+),\s*([^,]+),\s*([^,]+)')
df["location"] = df.apply(lambda row: f"{row['loc_1']} - {row['loc_2']} - {row['loc_3']}", axis=1)
df = df.drop(["loc_1","loc_2","loc_3"],axis=1)

In [3]:
def clean_numbers(text):
    return re.sub(r'\d', '', text)

df["location"] = df['location'].apply(clean_numbers)

In [4]:
df

,Date Text,Text Text,location
0,02.07.2023 01:49:53 UTC+03:00,"Перестрілка між російськими військовими, які в...",NE of Volodymyrivka - Volnovakha Raion - Donet...
1,02.07.2023 12:05:11 UTC+03:00,Український дрон робить скид на росіян в півде...,Oleshky - Kherson Oblast - Ukraine.
2,02.07.2023 13:48:35 UTC+03:00,Триочковий скид з ФПВ дрона від 11 бригади НГУ...,W of Novopokrovka - Zaporizhzhia Oblast - Ukra...
3,02.07.2023 18:46:07 UTC+03:00,Можливо російський літак випадково скинув ФАБ-...,Primorsko-Akhtarsk - Krasnodar Krai - Russia- ...
4,02.07.2023 22:18:56 UTC+03:00,Російський ФПВ дрон атакує українську позицію ...,S of Illinka - Dnipropetrovsk Oblast - Ukraine.
...,...,...,...
705,12.02.2024 16:11:53 UTC+03:00,429-й мотострілецький полк рф обстрілює україн...,S of Kamianske - Zaporizhzhia Oblast - Ukraine.
706,12.02.2024 18:25:05 UTC+03:00,18-та бригада армійської авіації некерованими ...,NW of Odradivka - Donetsk Oblast - Ukraine~.
707,13.02.2024 15:54:38 UTC+03:00,Збитий український Мі-24(?) 42-ю мотострілково...,NE of Robotyne - Zaporizhzhia Oblast - Ukraine.
708,13.02.2024 17:53:13 UTC+03:00,10-й танковий полк рф проводить евакуацію пошк...,S of Tsarska Ohota hotel - Avdiivka - Donetsk ...


## extract oblast data

In [5]:
df["location"] = df["location"].str.lower()

In [6]:
pattern = r'([a-zA-Z]+) oblast'

# Yeni bir sütun ekleyerek işlemi uygulayın
df["oblast"] = df['location'].apply(
    lambda x: re.findall(pattern, x)[0] if re.findall(pattern, x) else None
    )

## Extracting day month year and season datas from Date 

In [7]:
df['date'] = pd.to_datetime(df['Date Text'], utc=True, format='%d.%m.%Y %H:%M:%S UTC%z').dt.date
df['time'] = pd.to_datetime(df['Date Text'], utc=True, format='%d.%m.%Y %H:%M:%S UTC%z').dt.time
df['timezone'] = pd.to_datetime(df['Date Text'], utc=True, format='%d.%m.%Y %H:%M:%S UTC%z').dt.tz_convert(None).dt.tz_localize('UTC').dt.tz_convert('Europe/Istanbul').dt.tz_localize(None)

df['date'] = pd.to_datetime(df['date'])

# Yıl, ay ve gün bilgilerini ekleyin
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month
df['Day'] = df['date'].dt.day

# Mevsim bilgisini ekleyin
seasons = {
    1: 'Winter',
    2: 'Winter',
    3: 'Spring',
    4: 'Spring',
    5: 'Spring',
    6: 'Summer',
    7: 'Summer',
    8: 'Summer',
    9: 'Fall',
    10: 'Fall',
    11: 'Fall',
    12: 'Winter'
}
df['Season'] = df['Month'].map(seasons)

In [8]:
def siniflandir_am_pm(saat_str):
    saat = pd.to_datetime(saat_str, format="%H:%M:%S").time()
    if saat < pd.to_datetime("12:00:00", format="%H:%M:%S").time():
        return "AM"
    else:
        return "PM"

df["AM_PM"] = df["time"].apply(siniflandir_am_pm)

In [10]:
df.to_csv("../data/interim/data1.csv",index=False)